In [ ]:
import pandas as pd
import re

# ============================
# Carregar o arquivo
# ============================
caminho_arquivo = "imoveis_sao_caetano.xlsx"  # Altere se necessário
df = pd.read_excel(caminho_arquivo)

# ============================
# Funções de limpeza
# ============================

def extrair_maior_numero(texto, prefixo):
    try:
        valor = texto.replace(prefixo, "").replace("m²", "").replace("quartos", "").replace("banheiros", "").replace("vagas de garagem", "").strip()
        if "-" in valor:
            return int(valor.split("-")[1].strip())
        else:
            return int(valor)
    except:
        return None

def extrair_metragem(texto, prefixo):
    try:
        valor = texto.replace(prefixo, "").replace("m²", "").strip()
        return valor
    except:
        return None

def limpar_valor(valor):
    try:
        valor = valor.replace("A partir de R$", "").replace("R$", "").replace(".", "").replace(",", ".").strip()
        return float(valor)
    except:
        return None

def extrair_condominio(texto):
    if isinstance(texto, str):
        match = re.search(r'Cond(?:\.|omínio)?(?: a partir de)? R\$ ([\d\.]+)', texto)
        if match:
            return float(match.group(1).replace('.', ''))
    return None

def extrair_iptu(texto):
    if isinstance(texto, str):
        match = re.search(r'IPTU R\$ ([\d\.]+)', texto)
        if match:
            return float(match.group(1).replace('.', ''))
    return None

def extrair_bairro_cidade(texto):
    if isinstance(texto, str):
        match = re.search(r'em\n(.+?),\s*(.+)', texto)
        if match:
            return match.group(1).strip(), match.group(2).strip()
    return None, None

# ============================
# Preencher valores nulos com percentuais sobre Valor_R$
# ============================

df["Condominio_R$"] = df.apply(
    lambda row: round(row["Valor_R$"] * 0.0008, 2) if pd.isna(row["Condominio_R$"]) or row["Condominio_R$"] == 0 else row["Condominio_R$"],
    axis=1
).astype(float)

df["IPTU_R$"] = df.apply(
    lambda row: round(row["Valor_R$"] * 0.0004, 2) if pd.isna(row["IPTU_R$"]) or row["IPTU_R$"] == 0 else row["IPTU_R$"],
    axis=1
).astype(float)

# ============================
# Aplicar transformações
# ============================

df["Metragem"] = df["Metragem"].apply(lambda x: extrair_metragem(x, "Tamanho do imóvel\n"))
df["Quartos"] = df["Quartos"].apply(lambda x: extrair_maior_numero(x, "Quantidade de quartos\n"))
df["Banheiros"] = df["Banheiros"].apply(lambda x: extrair_maior_numero(x, "Quantidade de banheiros\n"))
df["Vagas"] = df["Vagas"].apply(lambda x: extrair_maior_numero(x, "Quantidade de vagas de garagem\n"))

df["Valor_R$"] = df["Valor"].apply(limpar_valor).astype(float)
df["Condominio_R$"] = df["IPTU  E CONDOMÍNIO"].apply(extrair_condominio)
df["IPTU_R$"] = df["IPTU  E CONDOMÍNIO"].apply(extrair_iptu)

df[["Bairro", "Cidade"]] = df["Bairro e Cidade"].apply(lambda x: pd.Series(extrair_bairro_cidade(x)))

# ============================
# Exportar apenas os campos tratados
# ============================
df_tratado = df[[ 
    "Cidade", "Bairro", "Nome da Rua", "Valor_R$", "Condominio_R$", "IPTU_R$", 
    "Metragem", "Quartos", "Banheiros", "Vagas"
]]

# Garantir tipos float
colunas_float = ["Valor_R$", "Condominio_R$", "IPTU_R$"]
df_tratado[colunas_float] = df_tratado[colunas_float].astype(float)

# Salvar o resultado
df_tratado.to_excel("imoveis_sao_caetano_tratado.xlsx", index=False)
print("Arquivo tratado salvo como: imoveis_sao_caetano_tratado.xlsx")


C:\Users\eduardo\AppData\Local\Temp\ipykernel_14364\236083044.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tratado[colunas_float] = df_tratado[colunas_float].astype(float)


Arquivo tratado salvo como: imoveis_sao_caetano_tratado.xlsx
